# Fabric Table Preview with PySpark

command to run the jupyter server for this notebook: runjupyter (in any folder)
equivalent to running this in /go-api:

```docker compose run --rm -p 8888:8888 serve jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root --notebook-dir=/home/ifrc```


In [2]:
import os
from pathlib import Path
from urllib.request import urlretrieve

POSTGRES_JDBC_VERSION = '42.7.4'
POSTGRES_JDBC_JAR = Path(f'/tmp/postgresql-{POSTGRES_JDBC_VERSION}.jar')
POSTGRES_JDBC_URL = (
    'https://repo1.maven.org/maven2/org/postgresql/postgresql/'
    f'{POSTGRES_JDBC_VERSION}/postgresql-{POSTGRES_JDBC_VERSION}.jar'
)

if not POSTGRES_JDBC_JAR.exists():
    urlretrieve(POSTGRES_JDBC_URL, POSTGRES_JDBC_JAR)

# Must be set before Spark JVM starts in this kernel
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--jars {POSTGRES_JDBC_JAR} pyspark-shell'

from pyspark.sql import SparkSession

In [3]:
# Stop the current Spark session so this cell can be rerun safely
active_spark = SparkSession.getActiveSession()
if active_spark is not None:
    active_spark.stop()

spark = (
    SparkSession.builder
    .appName('postgres-table-preview-notebook')
    .master('local[*]')
    .getOrCreate()
)

spark

26/03/06 19:14:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 19:14:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
# These are provided by docker compose env + .env when running notebook in `serve` container
PG_HOST = os.getenv('DJANGO_DB_HOST', 'db')
PG_PORT = os.getenv('DJANGO_DB_PORT', '5432')
PG_DB = os.environ['DJANGO_DB_NAME']
PG_USER = os.environ['DJANGO_DB_USER']
PG_PASSWORD = os.environ['DJANGO_DB_PASS']

jdbc_url = f'jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}'

# Read only table names in public schema to discover what you can load
table_list_query = """(
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
) t"""

tables_df = (
    spark.read.format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', table_list_query)
    .option('user', PG_USER)
    .option('password', PG_PASSWORD)
    .option('driver', 'org.postgresql.Driver')
    .load()
)

print('Total public tables:', tables_df.count())
tables_df.show(200, truncate=False)

Total public tables: 351
+---------------------------------------------------------+
|table_name                                               |
+---------------------------------------------------------+
|api_action                                               |
|api_actionstaken                                         |
|api_actionstaken_actions                                 |
|api_admin2                                               |
|api_admin2geoms                                          |
|api_admincontact                                         |
|api_adminkeyfigure                                       |
|api_adminlink                                            |
|api_appeal                                               |
|api_appealdocument                                       |
|api_appealdocumenttype                                   |
|api_appealfilter                                         |
|api_appealhistory                                        |
|api_authlog   

In [7]:
# Example: load one simple Django table from go-api/api/models.py
table_name = 'api_CleanedFrameworkAgreement'

df_sample = (
    spark.read.format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', table_name)
    .option('user', PG_USER)
    .option('password', PG_PASSWORD)
    .option('driver', 'org.postgresql.Driver')
    .load()
)

print('Table:', table_name)
df_sample.show(10, truncate=False)

Table: api_CleanedFrameworkAgreement
+----+------------+--------------+-------------------------------------+--------------------------------------+---------------+---------+--------------+----------------------------+------------------------+--------------+------------------------+---------+------------------+------------------------------+--------------------------+--------------------------+-----+-------------------+-------------------+
|id  |agreement_id|classification|default_agreement_line_effective_date|default_agreement_line_expiration_date|workflow_status|status   |price_per_unit|pa_line_procurement_category|vendor_name             |vendor_country|region_countries_covered|item_type|item_category     |item_service_short_description|created_at                |updated_at                |owner|vendor_valid_from  |vendor_valid_to    |
+----+------------+--------------+-------------------------------------+--------------------------------------+---------------+---------+------------